In [2]:
"""
ADD-ON: fixes issue 1 (augmentation tested only on corruptions it was trained on)
        and issue 2 (single seed, no confidence intervals / significance tests)

Run on Kaggle (GPU, Internet ON) with the same dataset as before.
  - Training augmentation uses ONLY the 5 original ("seen") corruptions.
  - Testing uses the 5 seen corruptions AND 5 NEW ("unseen") corruptions.
  - Every model/variant is trained with several seeds (same data split for all seeds).
  - Outputs: mean +/- std over seeds, bootstrap 95% CIs, McNemar tests, figures.
Results -> /kaggle/working/results_addon
"""
import os, glob, io, time, random, copy, itertools
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from sklearn.model_selection import train_test_split
from scipy.stats import binomtest

In [3]:
# ----------------------------- CONFIG -----------------------------
SPLIT_SEED = 42                 # keep identical to the first script -> same train/val/test split
SEEDS = [0, 1, 2]               # training seeds (use [0, 1] if you are short on time)
EPOCHS = 8
IMG, BATCH, LR = 224, 32, 5e-4
MODELS = ["mobilenet_v3_small", "shufflenet_v2_x1_0", "efficientnet_b0", "squeezenet1_1"]
N_BOOT = 5000
OUT = "/kaggle/working/results_addon"
CKPT = "/kaggle/working/ckpt"
os.makedirs(OUT, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"
print("Device:", DEVICE)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


Device: cuda


In [4]:
# ----------------------------- DATA (same split as the first script) -----------------------------
def find_root():
    for p in glob.glob("/kaggle/input/**/cardboard", recursive=True):
        if os.path.isdir(p):
            return os.path.dirname(p)
    raise FileNotFoundError("Dataset not found. Add it via 'Add Input'.")


ROOT = find_root()
CLASSES = sorted([d for d in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, d))])
paths, labels = [], []
for ci, c in enumerate(CLASSES):
    for f in glob.glob(os.path.join(ROOT, c, "*")):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            paths.append(f); labels.append(ci)
labels = np.array(labels)
print("Total images:", len(paths))
IMGS = [Image.open(p).convert("RGB").resize((IMG, IMG)) for p in paths]

idx_all = np.arange(len(IMGS))
trainval_idx, test_idx = train_test_split(idx_all, test_size=0.2, stratify=labels, random_state=SPLIT_SEED)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.1, stratify=labels[trainval_idx],
                                      random_state=SPLIT_SEED)
y_test = labels[test_idx]
print(f"Train {len(train_idx)} | Val {len(val_idx)} | Test {len(test_idx)}")
counts = np.bincount(y_test, minlength=len(CLASSES)).tolist()
print("Test class counts:", dict(zip(CLASSES, counts)))
if counts != [81, 100, 82, 119, 97, 27]:
    print("WARNING: test split differs from the first run (file order changed). "
          "Report ONLY the numbers from this add-on script in the paper.")

Total images: 2527
Train 1818 | Val 203 | Test 506
Test class counts: {'cardboard': 81, 'glass': 100, 'metal': 82, 'paper': 119, 'plastic': 97, 'trash': 27}


In [5]:
# ----------------------------- CORRUPTIONS -----------------------------
# --- SEEN (used for training augmentation AND testing; identical to the first script)
def c_blur(img, s):   return img.filter(ImageFilter.GaussianBlur([1, 2, 4][s - 1]))
def c_noise(img, s):
    a = np.asarray(img).astype(np.float32) + np.random.randn(224, 224, 3) * [10, 25, 50][s - 1]
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
def c_dark(img, s):   return ImageEnhance.Brightness(img).enhance([0.6, 0.35, 0.15][s - 1])
def c_jpeg(img, s):
    buf = io.BytesIO(); img.save(buf, "JPEG", quality=[30, 15, 5][s - 1]); buf.seek(0)
    return Image.open(buf).convert("RGB")
def c_lowres(img, s):
    f = [2, 4, 8][s - 1]; w = img.size[0]
    return img.resize((w // f, w // f)).resize((w, w))

SEEN = {"gaussian_blur": c_blur, "gaussian_noise": c_noise, "low_light": c_dark,
        "jpeg_compression": c_jpeg, "low_resolution": c_lowres}

# --- UNSEEN (NEVER used in training; test only)
def u_motion(img, s):
    k = [7, 15, 25][s - 1]
    a = np.asarray(img).astype(np.float32); w = a.shape[1]
    p = np.pad(a, ((0, 0), (k // 2, k // 2), (0, 0)), mode="edge")
    out = sum(p[:, d:d + w] for d in range(k)) / k
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))
def u_contrast(img, s):  return ImageEnhance.Contrast(img).enhance([0.5, 0.3, 0.15][s - 1])
def u_saltpepper(img, s):
    a = np.asarray(img).copy(); amt = [0.02, 0.06, 0.12][s - 1]
    m = np.random.rand(a.shape[0], a.shape[1])
    a[m < amt / 2] = 0; a[(m >= amt / 2) & (m < amt)] = 255
    return Image.fromarray(a)
def u_haze(img, s):
    t = [0.3, 0.5, 0.7][s - 1]
    a = np.asarray(img).astype(np.float32) * (1 - t) + 220.0 * t
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
def u_overexp(img, s):   return ImageEnhance.Brightness(img).enhance([1.6, 2.2, 3.0][s - 1])

UNSEEN = {"motion_blur": u_motion, "low_contrast": u_contrast, "salt_pepper": u_saltpepper,
          "haze": u_haze, "overexposure": u_overexp}


class RandCorrupt:
    """Training augmentation: ONLY the 5 seen corruptions."""
    def __init__(self, p=0.5): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        return random.choice(list(SEEN.values()))(img, random.randint(1, 3))


norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = T.Compose([T.RandomHorizontalFlip(), T.RandomRotation(15), T.ColorJitter(0.2, 0.2, 0.2),
                      T.ToTensor(), norm])
tf_eval = T.Compose([T.ToTensor(), norm])


class DS(Dataset):
    def __init__(self, idx, tf, corrupt=None): self.idx, self.tf, self.corrupt = idx, tf, corrupt
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; img = IMGS[j]
        if self.corrupt is not None: img = self.corrupt(img)
        return self.tf(img), int(labels[j])


def loader(idx, train=False, corrupt=None):
    return DataLoader(DS(idx, tf_train if train else tf_eval, corrupt), batch_size=BATCH, shuffle=train,
                      num_workers=4, pin_memory=USE_CUDA)

In [6]:
# ----------------------------- MODELS / TRAINING -----------------------------
def build(name, nc):
    if name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights="DEFAULT"); m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, nc)
    elif name == "shufflenet_v2_x1_0":
        m = models.shufflenet_v2_x1_0(weights="DEFAULT"); m.fc = nn.Linear(m.fc.in_features, nc)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights="DEFAULT"); m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, nc)
    elif name == "squeezenet1_1":
        m = models.squeezenet1_1(weights="DEFAULT"); m.classifier[1] = nn.Conv2d(512, nc, kernel_size=1)
    return m.to(DEVICE)


@torch.no_grad()
def val_acc(model, dl):
    model.eval(); c = n = 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        c += (model(x).argmax(1) == y).sum().item(); n += len(y)
    return c / n


def train_model(name, seed, corrupt_aug):
    set_seed(seed)
    model = build(name, len(CLASSES))
    dl_tr = loader(train_idx, True, RandCorrupt(0.5) if corrupt_aug else None); dl_va = loader(val_idx)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(dl_tr))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best, best_acc, hist = None, -1, []
    for ep in range(EPOCHS):
        model.train(); tl = n = 0
        for x, y in dl_tr:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=USE_CUDA):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y); n += len(y)
        va = val_acc(model, dl_va); hist.append({"epoch": ep + 1, "train_loss": tl / n, "val_acc": va})
        if va > best_acc: best_acc, best = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best); model.eval()
    return model, hist


In [7]:
# ----------------------------- STEP 1: TRAIN ALL (model x variant x seed) -----------------------------
print("\n=== STEP 1: Training ===")
STORE, HISTS = {}, []
t0 = time.time()
for seed in SEEDS:
    for name in MODELS:
        for variant, aug in [("standard", False), ("corruption_aug", True)]:
            key = (name, variant, seed); ck = f"{CKPT}/{name}_{variant}_{seed}.pt"
            if os.path.exists(ck):                       # resume support
                m = build(name, len(CLASSES)); m.load_state_dict(torch.load(ck, map_location=DEVICE)); m.eval()
            else:
                m, h = train_model(name, seed, aug); torch.save(m.state_dict(), ck)
                HISTS += [dict(model=name, variant=variant, seed=seed, **r) for r in h]
            STORE[key] = m
            print(f"[{(time.time()-t0)/60:5.1f} min] trained {key}")
if HISTS: pd.DataFrame(HISTS).to_csv(f"{OUT}/training_history_multiseed.csv", index=False)


=== STEP 1: Training ===
[  0.0 min] trained ('mobilenet_v3_small', 'standard', 0)
[  0.0 min] trained ('mobilenet_v3_small', 'corruption_aug', 0)
[  0.0 min] trained ('shufflenet_v2_x1_0', 'standard', 0)
[  0.0 min] trained ('shufflenet_v2_x1_0', 'corruption_aug', 0)
[  0.0 min] trained ('efficientnet_b0', 'standard', 0)
[  0.0 min] trained ('efficientnet_b0', 'corruption_aug', 0)
[  0.0 min] trained ('squeezenet1_1', 'standard', 0)
[  0.0 min] trained ('squeezenet1_1', 'corruption_aug', 0)
[  0.0 min] trained ('mobilenet_v3_small', 'standard', 1)
[  0.0 min] trained ('mobilenet_v3_small', 'corruption_aug', 1)
[  0.0 min] trained ('shufflenet_v2_x1_0', 'standard', 1)
[  0.0 min] trained ('shufflenet_v2_x1_0', 'corruption_aug', 1)
[  0.0 min] trained ('efficientnet_b0', 'standard', 1)
[  0.0 min] trained ('efficientnet_b0', 'corruption_aug', 1)
[  0.0 min] trained ('squeezenet1_1', 'standard', 1)
[  0.0 min] trained ('squeezenet1_1', 'corruption_aug', 1)
[  0.0 min] trained ('mobilene

In [8]:
# ----------------------------- STEP 2: EVALUATE ALL CONDITIONS -----------------------------
print("\n=== STEP 2: Evaluating on clean + seen + unseen corruptions ===")
CONDS = [("clean", "clean", 0, None)]
CONDS += [("seen", c, s, f) for c, f in SEEN.items() for s in (1, 2, 3)]
CONDS += [("unseen", c, s, f) for c, f in UNSEEN.items() for s in (1, 2, 3)]


@torch.no_grad()
def predict(model, X):
    out = []
    for i in range(0, len(X), 128):
        out.append(model(X[i:i + 128].to(DEVICE)).argmax(1).cpu())
    return torch.cat(out).numpy()


CORRECT, rows = {}, []
for cset, cname, sev, fn in CONDS:
    np.random.seed(1234); random.seed(1234)        # identical corrupted images for ALL models/seeds
    X = torch.stack([tf_eval(fn(IMGS[j], sev) if fn else IMGS[j]) for j in test_idx])
    for (name, variant, seed), m in STORE.items():
        corr = (predict(m, X) == y_test)
        CORRECT[(name, variant, seed, cname, sev)] = corr
        rows.append(dict(model=name, variant=variant, seed=seed, corruption_set=cset,
                         corruption=cname, severity=sev, accuracy=corr.mean()))
    print("evaluated", cset, cname, sev)
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/multiseed_results_long.csv", index=False)
np.savez_compressed(f"{OUT}/per_image_correctness.npz",
                    **{"|".join(map(str, k)): v for k, v in CORRECT.items()})


=== STEP 2: Evaluating on clean + seen + unseen corruptions ===
evaluated clean clean 0
evaluated seen gaussian_blur 1
evaluated seen gaussian_blur 2
evaluated seen gaussian_blur 3
evaluated seen gaussian_noise 1
evaluated seen gaussian_noise 2
evaluated seen gaussian_noise 3
evaluated seen low_light 1
evaluated seen low_light 2
evaluated seen low_light 3
evaluated seen jpeg_compression 1
evaluated seen jpeg_compression 2
evaluated seen jpeg_compression 3
evaluated seen low_resolution 1
evaluated seen low_resolution 2
evaluated seen low_resolution 3
evaluated unseen motion_blur 1
evaluated unseen motion_blur 2
evaluated unseen motion_blur 3
evaluated unseen low_contrast 1
evaluated unseen low_contrast 2
evaluated unseen low_contrast 3
evaluated unseen salt_pepper 1
evaluated unseen salt_pepper 2
evaluated unseen salt_pepper 3
evaluated unseen haze 1
evaluated unseen haze 2
evaluated unseen haze 3
evaluated unseen overexposure 1
evaluated unseen overexposure 2
evaluated unseen overexpo

In [9]:

# ----------------------------- STEP 3: SUMMARY TABLES (mean +/- std over seeds) -----------------------------
print("\n=== STEP 3: Summary tables ===")
per_seed = (res.groupby(["model", "variant", "seed", "corruption_set"]).accuracy.mean()
               .unstack("corruption_set").reset_index())
per_seed.to_csv(f"{OUT}/per_seed_summary.csv", index=False)

agg = per_seed.groupby(["model", "variant"]).agg(
    clean_mean=("clean", "mean"), clean_std=("clean", "std"),
    seen_mean=("seen", "mean"), seen_std=("seen", "std"),
    unseen_mean=("unseen", "mean"), unseen_std=("unseen", "std")).reset_index()
for k in ["clean", "seen", "unseen"]:
    agg[f"{k} (mean±std)"] = agg.apply(lambda r: f"{r[k+'_mean']*100:.2f} ± {r[k+'_std']*100:.2f}", axis=1)
agg.to_csv(f"{OUT}/summary_mean_std.csv", index=False)
print(agg[["model", "variant", "clean (mean±std)", "seen (mean±std)", "unseen (mean±std)"]].to_string(index=False))

per_c = (res[res.corruption_set != "clean"].groupby(["model", "variant", "seed", "corruption"]).accuracy.mean()
         .reset_index().groupby(["model", "variant", "corruption"]).accuracy.agg(["mean", "std"]).round(4))
per_c.to_csv(f"{OUT}/per_corruption_mean_std.csv")



=== STEP 3: Summary tables ===
             model        variant clean (mean±std) seen (mean±std) unseen (mean±std)
   efficientnet_b0 corruption_aug     94.14 ± 0.80    88.59 ± 1.23      79.54 ± 0.20
   efficientnet_b0       standard     94.27 ± 0.68    70.83 ± 1.72      74.91 ± 3.98
mobilenet_v3_small corruption_aug     90.25 ± 2.37    84.37 ± 1.64      74.71 ± 1.45
mobilenet_v3_small       standard     92.62 ± 1.26    72.16 ± 0.68      69.46 ± 0.34
shufflenet_v2_x1_0 corruption_aug     86.76 ± 1.89    80.07 ± 1.12      68.64 ± 1.04
shufflenet_v2_x1_0       standard     90.18 ± 0.57    63.48 ± 1.79      61.08 ± 1.42
     squeezenet1_1 corruption_aug     79.38 ± 1.48    71.98 ± 1.49      53.77 ± 1.62
     squeezenet1_1       standard     84.85 ± 1.99    50.83 ± 1.91      46.92 ± 2.89


In [10]:
# ----------------------------- STEP 4: STATISTICS -----------------------------
print("\n=== STEP 4: Bootstrap CIs and McNemar tests ===")
rng = np.random.RandomState(0)
COND_SETS = {"clean": [("clean", 0)],
             "seen": [(c, s) for c in SEEN for s in (1, 2, 3)],
             "unseen": [(c, s) for c in UNSEEN for s in (1, 2, 3)]}


def img_metric(name, variant, kind):
    """Per-test-image correctness averaged over seeds and conditions -> array of length N_test."""
    return np.mean([CORRECT[(name, variant, sd, c, s)] for sd in SEEDS for (c, s) in COND_SETS[kind]], axis=0)


def paired_boot(a, b):
    N = len(a); idx = rng.randint(0, N, (N_BOOT, N)); d = (a[idx] - b[idx]).mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5]); return (a - b).mean(), lo, hi


def boot_ci(a):
    N = len(a); idx = rng.randint(0, N, (N_BOOT, N)); m = a[idx].mean(1)
    return a.mean(), *np.percentile(m, [2.5, 97.5])


# (a) 95% CI for every model/variant/metric
ci_rows = []
for name in MODELS:
    for variant in ["standard", "corruption_aug"]:
        for kind in COND_SETS:
            m, lo, hi = boot_ci(img_metric(name, variant, kind))
            ci_rows.append(dict(model=name, variant=variant, metric=kind, mean=m, ci_low=lo, ci_high=hi))
pd.DataFrame(ci_rows).to_csv(f"{OUT}/bootstrap_ci_per_model.csv", index=False)

# (b) pairwise model comparison (within variant) + McNemar on clean predictions
pair_rows = []
for variant in ["standard", "corruption_aug"]:
    for a_, b_ in itertools.combinations(MODELS, 2):
        for kind in COND_SETS:
            d, lo, hi = paired_boot(img_metric(a_, variant, kind), img_metric(b_, variant, kind))
            row = dict(variant=variant, model_A=a_, model_B=b_, metric=kind, acc_diff_A_minus_B=d,
                       ci_low=lo, ci_high=hi, significant_95=(lo > 0 or hi < 0), mcnemar_median_p=np.nan)
            if kind == "clean":
                ps = []
                for sd in SEEDS:
                    ca, cb = CORRECT[(a_, variant, sd, "clean", 0)], CORRECT[(b_, variant, sd, "clean", 0)]
                    b01, c01 = int((ca & ~cb).sum()), int((~ca & cb).sum())
                    ps.append(binomtest(b01, b01 + c01, 0.5).pvalue if b01 + c01 > 0 else 1.0)
                row["mcnemar_median_p"] = float(np.median(ps))
            pair_rows.append(row)
pd.DataFrame(pair_rows).to_csv(f"{OUT}/pairwise_model_comparison.csv", index=False)

# (c) effect of corruption-aware augmentation (aug - standard) per model, incl. UNSEEN corruptions
gain_rows = []
for name in MODELS:
    for kind in COND_SETS:
        d, lo, hi = paired_boot(img_metric(name, "corruption_aug", kind), img_metric(name, "standard", kind))
        gain_rows.append(dict(model=name, metric=kind, gain_aug_minus_std=d, ci_low=lo, ci_high=hi,
                              significant_95=(lo > 0 or hi < 0)))
gain = pd.DataFrame(gain_rows); gain.to_csv(f"{OUT}/augmentation_gain_ci.csv", index=False)
print(gain.round(4).to_string(index=False))



=== STEP 4: Bootstrap CIs and McNemar tests ===
             model metric  gain_aug_minus_std  ci_low  ci_high  significant_95
mobilenet_v3_small  clean             -0.0237 -0.0375  -0.0112            True
mobilenet_v3_small   seen              0.1221  0.1106   0.1339            True
mobilenet_v3_small unseen              0.0525  0.0417   0.0634            True
shufflenet_v2_x1_0  clean             -0.0343 -0.0520  -0.0178            True
shufflenet_v2_x1_0   seen              0.1659  0.1502   0.1817            True
shufflenet_v2_x1_0 unseen              0.0757  0.0638   0.0877            True
   efficientnet_b0  clean             -0.0013 -0.0152   0.0112           False
   efficientnet_b0   seen              0.1776  0.1677   0.1878            True
   efficientnet_b0 unseen              0.0463  0.0372   0.0556            True
     squeezenet1_1  clean             -0.0547 -0.0751  -0.0356            True
     squeezenet1_1   seen              0.2115  0.1916   0.2311            True
   

In [11]:
# ----------------------------- STEP 5: FIGURES -----------------------------
print("\n=== STEP 5: Figures ===")
COLORS = dict(zip(MODELS, ["tab:blue", "tab:orange", "tab:green", "tab:red"]))


def curve_figure(corr_dict, fname, title):
    fig, ax = plt.subplots(1, len(corr_dict), figsize=(4.2 * len(corr_dict), 4), sharey=True)
    ax = np.atleast_1d(ax)
    for k, cname in enumerate(corr_dict):
        for name in MODELS:
            for variant, ls in [("standard", "-"), ("corruption_aug", "--")]:
                means, stds = [], []
                for sev in (0, 1, 2, 3):
                    cn, sv = ("clean", 0) if sev == 0 else (cname, sev)
                    vals = [CORRECT[(name, variant, sd, cn, sv)].mean() for sd in SEEDS]
                    means.append(np.mean(vals)); stds.append(np.std(vals, ddof=1) if len(vals) > 1 else 0)
                means, stds = np.array(means), np.array(stds)
                ax[k].plot(range(4), means, ls, marker="o", color=COLORS[name],
                           label=f"{name} ({variant})" if k == 0 else None)
                ax[k].fill_between(range(4), means - stds, means + stds, color=COLORS[name], alpha=0.12)
        ax[k].set_title(cname); ax[k].set_xlabel("Severity"); ax[k].set_xticks(range(4))
    ax[0].set_ylabel("Test accuracy (mean ± std over seeds)"); ax[0].legend(fontsize=6)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=200); plt.close()


curve_figure(UNSEEN, "robustness_curves_UNSEEN.png", "Held-out (unseen) corruptions - never used in training")
curve_figure(SEEN, "robustness_curves_SEEN.png", "Seen corruptions - used in training augmentation")

fig, ax = plt.subplots(figsize=(9, 4.5)); w = 0.2; x = np.arange(len(MODELS))
bars = [("standard", "seen", "Standard - seen", "#9ecae1"), ("corruption_aug", "seen", "Aug - seen", "#3182bd"),
        ("standard", "unseen", "Standard - unseen", "#fdae6b"), ("corruption_aug", "unseen", "Aug - unseen", "#e6550d")]
for i, (v, k, lab, col) in enumerate(bars):
    d = agg[agg.variant == v].set_index("model").loc[MODELS]
    ax.bar(x + (i - 1.5) * w, d[f"{k}_mean"], w, yerr=d[f"{k}_std"], color=col, label=lab, capsize=3)
ax.set_xticks(x); ax.set_xticklabels(MODELS, rotation=15); ax.set_ylabel("Mean accuracy under corruption")
ax.set_ylim(0, 1); ax.legend(fontsize=8, ncol=2); plt.tight_layout()
plt.savefig(f"{OUT}/seen_vs_unseen_bar.png", dpi=200); plt.close()

print("\nALL DONE ->", OUT); print(sorted(os.listdir(OUT)))



=== STEP 5: Figures ===

ALL DONE -> /kaggle/working/results_addon
['augmentation_gain_ci.csv', 'bootstrap_ci_per_model.csv', 'multiseed_results_long.csv', 'pairwise_model_comparison.csv', 'per_corruption_mean_std.csv', 'per_image_correctness.npz', 'per_seed_summary.csv', 'robustness_curves_SEEN.png', 'robustness_curves_UNSEEN.png', 'seen_vs_unseen_bar.png', 'summary_mean_std.csv', 'training_history_multiseed.csv']
